# serve_vllm: Colab T4에서 vLLM 서버 + ngrok

`generate_problem` 평가용 로컬 모델을 OpenAI 호환 API로 띄운다.
base 모델과 학습한 LoRA 어댑터를 **한 서버에서 같이** 서빙하므로, 요청의 `model` 값만 바꿔 비교할 수 있다.

**준비 (처음 한 번)**
- 런타임 → 런타임 유형 변경 → **T4 GPU**
- 비밀값 3개를 둘 중 한 곳에 둔다
  - Colab 웹: 왼쪽 🔑 Secrets에 등록 (노트북 액세스 켜기)
  - VS Code 확장(Secrets 미지원): 저장소 루트의 `.env.colab`에 `이름=값`으로 적는다 (gitignore 대상). 4번 셀 입력창에 `base64 -i .env.colab | tr -d '\n' | pbcopy`로 복사한 값을 붙여넣는다
  - `HF_TOKEN`: Hugging Face 토큰 (read면 충분, 비공개 어댑터 repo를 읽을 수 있어야 함)
  - `NGROK_AUTHTOKEN`: ngrok 대시보드의 authtoken
  - `VLLM_API_KEY`: 서버 접근 키. Mac에서 `python3 -c "import secrets; print(secrets.token_hex(32))"`로 만들고, **같은 값을 Mac `.env`에도** 넣는다

**사용법**: 1번 설정 셀만 고치고 위에서부터 차례로 실행한다. 끝나면 맨 아래 정리 셀을 실행한다.

⚠️ 공유하거나 커밋하기 전에 **수정 → 모든 출력 지우기**. 키는 출력하지 않도록 만들어 두었지만 로그에 남을 수 있다.
⚠️ 학습 노트북과 같은 런타임에서 돌리지 않는다 (GPU 하나, 패키지 버전 충돌).

## 1. 설정

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B"   # 서빙할 base 모델 (HF repo id)

# 학습한 LoRA 어댑터. {요청에 쓸 이름: HF repo id}. 없으면 빈 dict로 둔다
LORA_ADAPTERS = {
    # "qwen3-4b-genprob-v1": "your-hf-id/qwen3-4b-genprob-lora-v1",
}
MAX_LORA_RANK = 16           # 학습 노트북 LORA_R 이상

MAX_MODEL_LEN = 8192         # 프롬프트(스키마 포함 약 1.1k) + 출력(약 1k) 여유
NGROK_DOMAIN = None          # 고정 도메인 "xxxx.ngrok-free.app". None이면 랜덤 URL
EXTRA_ARGS = []              # 모델별 추가 옵션. 비교하는 모델끼리는 같게 맞춘다

VLLM_VERSION = "0.31.0"      # T4에서 검증한 버전. 바꾸면 기동부터 다시 확인
PORT = 8000

## 2. GPU 확인

In [ ]:
!nvidia-smi

## 3. 설치

설치 후 "런타임을 다시 시작하라"는 안내가 뜨면 재시작하고 **1번 설정 셀부터** 다시 실행한다.

In [ ]:
!pip install -q "vllm=={VLLM_VERSION}" pyngrok

import vllm, torch
print(vllm.__version__, torch.__version__, torch.cuda.is_available())

## 4. 비밀값 (출력하지 않는다)

In [ ]:
import base64
import os
from getpass import getpass


def load_secrets(names):
    """Colab Secrets에서 읽는다. 안 되면(VS Code 확장) 로컬 .env.colab 묶음을 한 번 붙여넣어 받는다.

    Mac: base64 -i .env.colab | tr -d '\n' | pbcopy  → 입력창에 붙여넣기
    값은 화면과 노트북에 남지 않는다.
    """
    try:
        from google.colab import userdata

        values = {name: userdata.get(name) for name in names}
    except Exception:
        bundle = base64.b64decode(getpass(".env.colab (base64) 붙여넣기: ")).decode()
        values = {}
        for line in bundle.splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                values[key.strip()] = value.strip().strip('"').strip("'")
    missing = [name for name in names if not values.get(name)]
    if missing:
        raise ValueError(f"비밀값이 없습니다: {missing}")
    for name in names:
        os.environ[name] = values[name]  # 자식 프로세스(vllm 등)까지 전달된다


load_secrets(["HF_TOKEN", "NGROK_AUTHTOKEN", "VLLM_API_KEY"])

# '-'로 시작하면 vllm 인자 파서가 값이 아니라 옵션으로 읽는다
if os.environ["VLLM_API_KEY"].startswith("-"):
    raise ValueError("VLLM_API_KEY가 '-'로 시작합니다. token_hex로 다시 만드세요")
print("secrets OK")

## 5. LoRA 어댑터 내려받기 (없으면 건너뛰어도 된다)

In [ ]:
from huggingface_hub import snapshot_download

lora_paths = {name: snapshot_download(repo) for name, repo in LORA_ADAPTERS.items()}
print(lora_paths or "어댑터 없음 (base만 서빙)")

## 6. 서버 시작

남아 있는 서버를 먼저 정리한다. 두 개가 겹치면 포트 충돌로 새 서버가 죽고 로그도 뒤섞인다.

In [ ]:
import subprocess
import time


def stop_server():
    """이 노트북이 띄운 서버와, 다른 셀에서 띄운 vllm까지 정리한다."""
    if "server" in globals() and server.poll() is None:
        server.terminate()
        server.wait(timeout=60)
    subprocess.run(["pkill", "-f", "vllm serve"])
    time.sleep(3)


stop_server()

cmd = [
    "vllm", "serve", MODEL_ID,
    "--dtype", "half",                      # T4는 bf16 미지원
    "--max-model-len", str(MAX_MODEL_LEN),
    "--port", str(PORT),
    f"--api-key={os.environ['VLLM_API_KEY']}",
]
if lora_paths:
    cmd += ["--enable-lora", "--max-lora-rank", str(MAX_LORA_RANK), "--lora-modules"]
    cmd += [f"{name}={path}" for name, path in lora_paths.items()]
cmd += EXTRA_ARGS

log = open("vllm.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print("started pid", server.pid)
print(" ".join("--api-key=***" if c.startswith("--api-key") else c for c in cmd))

## 7. 준비될 때까지 대기 (첫 실행은 다운로드·컴파일로 수 분)

In [ ]:
import requests


def tail_log(n=40):
    print(subprocess.run(["tail", "-n", str(n), "vllm.log"], capture_output=True, text=True).stdout)


def wait_ready(timeout_s=900):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if server.poll() is not None:
            print(f"서버가 종료됨 (exit {server.returncode})")
            tail_log()
            return False
        try:
            if requests.get(f"http://localhost:{PORT}/health", timeout=2).ok:
                print("READY")
                return True
        except requests.ConnectionError:
            pass
        time.sleep(5)
    print(f"{timeout_s}초 안에 준비되지 않음")
    tail_log()
    return False


wait_ready()

## 8. 스모크 테스트 (서빙 중인 모델마다 한 번씩)

In [ ]:
HEADERS = {"Authorization": f"Bearer {os.environ['VLLM_API_KEY']}"}
API = f"http://localhost:{PORT}/v1"

served = [m["id"] for m in requests.get(f"{API}/models", headers=HEADERS).json()["data"]]
print("served models:", served)

for name in served:
    resp = requests.post(
        f"{API}/chat/completions",
        headers=HEADERS,
        json={
            "model": name,
            "messages": [{"role": "user", "content": "1부터 10까지 더하면? 숫자만 답해."}],
            "max_tokens": 50,
            "chat_template_kwargs": {"enable_thinking": False},
        },
        timeout=120,
    )
    resp.raise_for_status()
    print(f"{name} → {resp.json()['choices'][0]['message']['content'].strip()}")

## 9. ngrok 터널

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
ngrok.kill()  # 이전 터널 정리
tunnel = ngrok.connect(PORT, **({"domain": NGROK_DOMAIN} if NGROK_DOMAIN else {}))
print("VLLM_URL =", tunnel.public_url)

## 10. Mac에서 쓰기

저장소 `.env`에 추가한다 (`.env`는 커밋되지 않는다).

```
VLLM_URL=<9번 셀에 나온 URL>
VLLM_API_KEY=<Colab Secrets에 넣은 값과 같은 값>
```

요청의 `model` 값은 8번 셀에 나온 이름 중 하나를 쓴다 (base는 `MODEL_ID`, 학습본은 `LORA_ADAPTERS`의 이름).

Mac에서 확인:

```bash
curl -s -o /dev/null -w "%{http_code}\n" $VLLM_URL/v1/models   # 키 없이 → 401
```

## 로그 확인 (문제가 생겼을 때)

In [ ]:
!grep -a -i -E "error|attention backend|Model loading took|KV cache size|lora" vllm.log | tail -20

## 정리 (작업이 끝나면)

In [ ]:
stop_server()
ngrok.kill()
print("stopped")